# AI Learning — Student LLM on Kaggle

This notebook demonstrates a complete implementation of a byte-level transformer model for incremental learning on educational content.

## Section 1: Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

import math
import json
from dataclasses import dataclass
from typing import List, Tuple

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Section 2: Tokenizer (Byte-Level)

Fixed 260-token vocabulary: 4 special tokens + 256 UTF-8 bytes.

In [ ]:
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
BYTE_OFFSET = 4
VOCAB_SIZE = 260

class ByteTokenizer:
    """Fixed-vocabulary byte-level tokenizer."""
    
    PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
    VOCAB_SIZE = 260
    SPECIAL_TOKENS = {0: '<pad>', 1: '<unk>', 2: '<bos>', 3: '<eos>'}
    
    def encode(self, text: str, add_special: bool = False) -> List[int]:
        ids = [b + BYTE_OFFSET for b in text.encode('utf-8')]
        if add_special:
            ids = [BOS_ID] + ids + [EOS_ID]
        return ids
    
    def decode(self, ids: List[int], skip_special: bool = True) -> str:
        if skip_special:
            ids = [i for i in ids if i not in self.SPECIAL_TOKENS]
        raw_bytes = bytes(i - BYTE_OFFSET for i in ids if BYTE_OFFSET <= i < BYTE_OFFSET + 256)
        return raw_bytes.decode('utf-8', errors='replace')

tok = ByteTokenizer()
print(f'Tokenizer ready: vocab_size={tok.VOCAB_SIZE}')

In [ ]:
text = 'Hello, world!'
ids = tok.encode(text, add_special=True)
decoded = tok.decode(ids)
print(f'Text: {text}')
print(f'IDs: {ids}')
print(f'Decoded: {decoded}')
print(f'Match: {decoded == text}')

## Section 3: Model Architecture

Decoder-only transformer: 6 layers, 8 heads, 256 d_model

In [ ]:
@dataclass
class StudentConfig:
    vocab_size: int = 260
    d_model: int = 256
    num_layers: int = 6
    num_heads: int = 8
    d_ff: int = 1024
    dropout: float = 0.1
    max_seq_len: int = 2048
    
    def __post_init__(self):
        self.d_head = self.d_model // self.num_heads

config = StudentConfig()
print(f'Config: {config}')

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config: StudentConfig):
        super().__init__()
        self.num_heads = config.num_heads
        self.d_head = config.d_head
        
        self.q_proj = nn.Linear(config.d_model, config.d_model)
        self.k_proj = nn.Linear(config.d_model, config.d_model)
        self.v_proj = nn.Linear(config.d_model, config.d_model)
        self.out_proj = nn.Linear(config.d_model, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        B, T, C = x.shape
        Q = self.q_proj(x).view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        K = self.k_proj(x).view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        V = self.v_proj(x).view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        causal_mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        scores = scores.masked_fill(causal_mask == 0, float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

class FeedForward(nn.Module):
    def __init__(self, config: StudentConfig):
        super().__init__()
        self.fc1 = nn.Linear(config.d_model, config.d_ff)
        self.fc2 = nn.Linear(config.d_ff, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
    
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, config: StudentConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.ff = FeedForward(config)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

print('Attention layers defined')

In [ ]:
class StudentTransformer(nn.Module):
    def __init__(self, config: StudentConfig):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.num_layers)])
        self.ln_final = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size)
        self.lm_head.weight = self.embed.weight
    
    def forward(self, input_ids):
        B, T = input_ids.shape
        pos_ids = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.embed(input_ids) + self.pos_embed(pos_ids)
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_final(x)
        return self.lm_head(x)
    
    def generate(self, input_ids, max_tokens=100, temperature=1.0):
        for _ in range(max_tokens):
            if input_ids.shape[1] >= self.config.max_seq_len:
                break
            logits = self.forward(input_ids)[:, -1, :]
            logits = logits / temperature
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids
    
    def num_params(self):
        return sum(p.numel() for p in self.parameters())

model = StudentTransformer(config).to(device)
print(f'Model: {model.num_params():,} parameters')

## Section 4: Training

Support for document learning (next-token prediction) and assessment learning (supervised fine-tuning).

In [ ]:
@dataclass
class TrainerConfig:
    lr: float = 1e-3
    weight_decay: float = 1e-4
    betas: Tuple[float, float] = (0.9, 0.95)
    doc_lr_scale: float = 1.0
    assess_lr_scale: float = 0.5
    min_loss_scale: float = 0.05

class Trainer:
    def __init__(self, model, config: TrainerConfig, device='cpu'):
        self.model = model
        self.config = config
        self.device = device
        self.optimizer = AdamW(model.parameters(), lr=config.lr, betas=config.betas, weight_decay=config.weight_decay)
        self.history = []
    
    def learn_document(self, text: str, tokenizer):
        ids = tokenizer.encode(text, add_special=True)
        ids = torch.tensor(ids, dtype=torch.long, device=self.device).unsqueeze(0)
        
        loss_total = 0.0
        num_chunks = 0
        
        for i in range(0, max(len(ids[0]) - 1, 1), 256):
            chunk = ids[:, i:i+512]
            if chunk.shape[1] < 2:
                continue
            
            input_ids = chunk[:, :-1]
            target_ids = chunk[:, 1:]
            
            logits = self.model(input_ids)
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), target_ids.reshape(-1))
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            loss_total += loss.item()
            num_chunks += 1
        
        avg_loss = loss_total / max(num_chunks, 1)
        self.history.append({'type': 'document', 'loss': avg_loss})
        return avg_loss
    
    def learn_from_assessment(self, question: str, answer: str, score: float, max_score: float = 10.0):
        tokenizer = ByteTokenizer()
        q_ids = tokenizer.encode(question, add_special=False)
        a_ids = tokenizer.encode(answer, add_special=False)
        ids = q_ids + a_ids
        
        ids_tensor = torch.tensor(ids, dtype=torch.long, device=self.device).unsqueeze(0)
        
        input_ids = ids_tensor[:, :-1]
        target_ids = ids_tensor[:, 1:]
        
        logits = self.model(input_ids)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), target_ids.reshape(-1))
        loss_scale = max(1.0 - (score / max_score), self.config.min_loss_scale)
        scaled_loss = loss * loss_scale
        
        self.optimizer.zero_grad()
        scaled_loss.backward()
        self.optimizer.step()
        
        self.history.append({'type': 'assessment', 'loss': loss.item(), 'score': score})
        return loss.item()

trainer_config = TrainerConfig()
trainer = Trainer(model, trainer_config, device=device)
print('Trainer ready')

## Section 5: Learning from Documents

In [ ]:
sample_text = '''Machine learning is a branch of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms that can process data and make predictions.'''

print('Training on sample text...')
loss = trainer.learn_document(sample_text, tok)
print(f'Learning complete. Loss: {loss:.4f}')

## Section 6: Generating Responses

In [ ]:
def generate_response(prompt: str, max_length: int = 50):
    ids = tok.encode(prompt, add_special=False)
    input_tensor = torch.tensor([ids], device=device)
    with torch.no_grad():
        generated = model.generate(input_tensor, max_tokens=max_length)
    return tok.decode(generated[0].cpu().tolist(), skip_special=True)

prompt = 'Machine learning is'
response = generate_response(prompt, max_length=30)
print(f'Prompt: {prompt}')
print(f'Response: {response}')

## Section 7: Summary

Complete implementation of byte-level transformer for educational learning. The model supports document learning and assessment-based fine-tuning.